# IHK Gewerbedaten

In [0]:
%sh cat ../requirements.txt

In [0]:
#load packages
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Loading & Basic Cleaning

In [0]:
#load dataset
df = pd.read_csv("/Workspace/my_project/raw/2026_06_IHK_Berlin_Gewerbedaten.csv")

In [0]:
df.sample(3)

In [0]:
df.info()

In [0]:
missing_percent = round((df.isnull().sum() / len(df)) * 100,2)
missing_percent = missing_percent.sort_values(ascending=False)
print(missing_percent)

There are some missing values, most of them concerning the business age / descriptions. Since those make up less than 1% of the data, let's delete them.

In [0]:
#drop NA values
df.dropna(inplace=True)
df.shape

In [0]:
#check for duplicates
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

## Exploration

In [0]:
df.columns

In [0]:
#rows for each PLR
df.groupby('Planungsraum').size()

In [0]:
#how many employees
df['employees_range'].value_counts()

Es gibt viele Businesses, wo die employees_range unbekannt ist. Da müssen wir schauen, wie wir das lösen. Gleichzeitig gibt es auch Unternehmen, die keine Angestellten haben. Also inhabergeführte Gewerbe, evtl. auch Familienunternehmen? 

In [0]:
#business age
df.groupby("Planungsraum")['business_age'].median().sort_values(ascending = False)

### Gewerbearten

In [0]:
#Übersicht über die verschiedenen Kategorien
df[['nace_desc','ihk_branch_desc',  'branch_top_level_desc']].head(5)

In [0]:
#Anteil der business types berechnen
business_mix = pd.crosstab(
    df['Planungsraum'],
    df['business_type'],
    normalize='index'
)

In [0]:
business_mix.sample(4)

In [0]:
#Übersicht Kleingewerbetreibender vs. im Handelsregister eingetragen

# separate groups
klein = df[df['business_type'] == 'Kleingewerbetreibender']
register = df[df['business_type'] == 'im Handelsregister eingetragen']

# side by side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# plot 1
axes[0].scatter(
    klein['longitude'],
    klein['latitude'],
    s=1,
    alpha=0.3,
    color='steelblue' 
)
axes[0].set_title("Kleingewerbetreibender")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# plot 2
axes[1].scatter(
    register['longitude'],
    register['latitude'],
    s=1,
    alpha=0.3,
    color='orange' 
)
axes[1].set_title("Im Handelsregister eingetragen")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [0]:
df['branch_top_level_desc'].value_counts().head(15)

In [0]:
#Übersicht über Gastronomie: 
df.loc[df['branch_top_level_desc'] == 'Gastronomie'].groupby('Planungsraum')['ihk_branch_desc'].count().sort_values(ascending = False)

In [0]:
df.loc[df['branch_top_level_desc'] == 'Gastronomie']["nace_desc"].value_counts()

In [0]:
df.loc[df['branch_top_level_desc'] == 'Gastronomie']["ihk_branch_desc"].value_counts()

- Hier wichtig wahrscheinlich: Restaurants mit Bedienung, Cafés, Bars 
- Wie gehen wir mit Mischkategorien um?

In [0]:
df.loc[df['branch_top_level_desc'] == 'Einzelhandel']["ihk_branch_desc"].value_counts().head(15)

Hier evtl. wichtig: Einzelhandel mit..
- Bekleidung (Boutiquen?) - sicher auch fast Fashion mit drin
- Uhren/Schmuck? 
- Kunst- und Kulturerzeugnissen
- Bücher?

In [0]:
df.loc[df['branch_top_level_desc'] == 'Einzelhandel']["nace_desc"].value_counts().head(15)

In [0]:
df.loc[df['branch_top_level_desc'] == 'Erbringung von Dienstleistungen des Sports, der Unterhaltung und der Erholung']["ihk_branch_desc"].value_counts().head(15)

Hier wahrscheinlich wichtig: 
- Fitnesszentren 
- Erbringung von Dienstleistungen des Sports

In [0]:
df['branch_top_level_desc'].value_counts().sort_values(ascending = True).head(15)

In [0]:
#hotels: nace codes start with 55 
accommodation = df[df['nace_id'].astype(str).str.startswith('55')]
accommodation["branch_top_level_desc"].value_counts()

In [0]:
df.loc[df['branch_top_level_desc'] == 'Beherbergung']["ihk_branch_desc"].value_counts().head(15)